# Interactive Boundary Visualization

Explore semantic decision boundaries and their impact on trajectory prediction.

**Key finding:** Boundaries are fuzzy (not crisp). Same-class and cross-class distance distributions overlap. ADE impact does NOT correlate with embedding distance (r=-0.018).

## 1. Setup & Data Loading

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import ndimage
from sklearn.neighbors import KNeighborsClassifier
import ipywidgets as widgets
from IPython.display import display, HTML

# Design system
COLORS = {
    "bg": "#fafafa",
    "text": "#1a1a1a",
    "grid": "#e0e0e0",
    "accent": "#667eea",
}

# ADE discrete classes
ADE_BINS = [0, 1.0, 2.5, 5.0, float("inf")]
ADE_LABELS = ["low", "medium", "high", "critical"]
ADE_COLORS = {"low": "#2ecc71", "medium": "#f1c40f", "high": "#e67e22", "critical": "#e74c3c"}

CLASSIFICATION_KEYS = [
    "weather", "time_of_day", "depth_complexity",
    "occlusion_level", "road_type", "required_action"
]

# Muted categorical palette for semantic keys
KEY_COLORS = px.colors.qualitative.Set2[:6]


def classify_ade(ade: float) -> str:
    """Discretize ADE into classes."""
    if pd.isna(ade):
        return None
    if ade < 1.0:
        return "low"
    elif ade < 2.5:
        return "medium"
    elif ade < 5.0:
        return "high"
    else:
        return "critical"


def plotly_layout(title: str = "", height: int = 500) -> dict:
    """Standard layout for all plots."""
    return dict(
        title=dict(text=title, x=0.5, font=dict(size=14, color=COLORS["text"])),
        paper_bgcolor=COLORS["bg"],
        plot_bgcolor=COLORS["bg"],
        font=dict(family="Inter, system-ui, sans-serif", color=COLORS["text"]),
        height=height,
        margin=dict(l=60, r=40, t=60, b=60),
    )


print("Design system loaded.")

Design system loaded.


In [2]:
# Data paths
REPO_ROOT = Path.cwd().parent.parent
DATA_DIR = REPO_ROOT / "data" / "pipeline"

# Load data
scenes = pd.read_parquet(DATA_DIR / "scenes.parquet")
pairs = pd.read_parquet(DATA_DIR / "results" / "pairs.parquet")
embeddings = np.load(DATA_DIR / "embeddings.npz")["embeddings"]

with open(DATA_DIR / "results" / "stability_map.json") as f:
    stability = json.load(f)

with open(DATA_DIR / "results" / "summary.json") as f:
    summary = json.load(f)

# Load or compute 3D UMAP
umap_path = Path("/tmp/emb_3d.npy")
if umap_path.exists():
    umap_3d = np.load(umap_path)
    print(f"Loaded pre-computed 3D UMAP: {umap_3d.shape}")
else:
    from umap import UMAP
    reducer = UMAP(n_components=3, n_neighbors=15, min_dist=0.1, random_state=42)
    umap_3d = reducer.fit_transform(embeddings)
    np.save(umap_path, umap_3d)
    print(f"Computed 3D UMAP: {umap_3d.shape}")

# Add UMAP coords and ADE class to scenes
scenes["umap_x"] = umap_3d[:, 0]
scenes["umap_y"] = umap_3d[:, 1]
scenes["umap_z"] = umap_3d[:, 2]
scenes["ade_class"] = scenes["ade"].apply(classify_ade)

# Add ADE classes to pairs
pairs["ade_class_a"] = pairs["ade_a"].apply(classify_ade)
pairs["ade_class_b"] = pairs["ade_b"].apply(classify_ade)
pairs["ade_class_changed"] = pairs["ade_class_a"] != pairs["ade_class_b"]

print(f"Loaded {len(scenes):,} scenes, {len(pairs):,} pairs")
print(f"Scenes with ADE: {scenes['ade'].notna().sum():,}")
print(f"Pairs with ADE: {pairs['rel_delta_ade'].notna().sum():,}")

Loaded pre-computed 3D UMAP: (2647, 3)
Loaded 2,647 scenes, 13,043 pairs
Scenes with ADE: 947
Pairs with ADE: 4,270


## 2. Embedding Space Explorer (3D)

In [3]:
def create_3d_explorer(scenes_df: pd.DataFrame, color_by: str = "ade_class") -> go.FigureWidget:
    """Create interactive 3D embedding space visualization."""
    df = scenes_df.copy()
    
    # Point size based on ADE (larger = higher error)
    ade_for_size = df["ade"].fillna(df["ade"].median())
    sizes = 3 + (ade_for_size / ade_for_size.max()) * 8
    
    # Build hover text
    hover_text = []
    for _, row in df.iterrows():
        text = f"<b>{row['clip_id'][:8]}...</b><br>"
        for key in CLASSIFICATION_KEYS:
            text += f"{key}: {row[key]}<br>"
        if pd.notna(row["ade"]):
            text += f"<b>ADE: {row['ade']:.3f} ({row['ade_class']})</b><br>"
        text += f"Anchor: {row['is_anchor']}"
        hover_text.append(text)
    df["hover"] = hover_text
    
    # Color mapping
    if color_by == "ade_class":
        df["color"] = df["ade_class"].map(ADE_COLORS).fillna("#cccccc")
        color_discrete_map = ADE_COLORS
    elif color_by == "is_anchor":
        df["color"] = df["is_anchor"].map({True: "#e74c3c", False: "#3498db"})
        color_discrete_map = {True: "#e74c3c", False: "#3498db"}
    elif color_by == "label_confidence":
        df["color"] = df["label_confidence"].fillna(0)
        color_discrete_map = None
    elif color_by in CLASSIFICATION_KEYS:
        unique_vals = df[color_by].unique()
        color_discrete_map = {v: px.colors.qualitative.Set2[i % 8] for i, v in enumerate(unique_vals)}
        df["color"] = df[color_by].map(color_discrete_map)
    else:
        df["color"] = "#3498db"
        color_discrete_map = None
    
    # Create scatter
    fig = go.FigureWidget()
    
    # Main points
    if color_by == "label_confidence":
        fig.add_trace(go.Scatter3d(
            x=df["umap_x"], y=df["umap_y"], z=df["umap_z"],
            mode="markers",
            marker=dict(
                size=sizes,
                color=df["label_confidence"].fillna(0),
                colorscale="Viridis",
                colorbar=dict(title="Confidence"),
            ),
            text=df["hover"],
            hoverinfo="text",
            name="Scenes",
        ))
    else:
        # Group by color value for legend
        for label in df[color_by].unique():
            mask = df[color_by] == label
            subset = df[mask]
            color = subset["color"].iloc[0] if len(subset) > 0 else "#cccccc"
            fig.add_trace(go.Scatter3d(
                x=subset["umap_x"], y=subset["umap_y"], z=subset["umap_z"],
                mode="markers",
                marker=dict(size=sizes[mask].tolist(), color=color, opacity=0.7),
                text=subset["hover"],
                hoverinfo="text",
                name=str(label) if pd.notna(label) else "N/A",
            ))
    
    # Highlight anchors with distinct marker
    anchors = df[df["is_anchor"] == True]
    if len(anchors) > 0 and color_by != "is_anchor":
        fig.add_trace(go.Scatter3d(
            x=anchors["umap_x"], y=anchors["umap_y"], z=anchors["umap_z"],
            mode="markers",
            marker=dict(size=6, color="rgba(0,0,0,0)", line=dict(color="#1a1a1a", width=2)),
            text=anchors["hover"],
            hoverinfo="text",
            name="Anchors (VLM)",
        ))
    
    fig.update_layout(
        **plotly_layout(f"Embedding Space (colored by {color_by})", height=600),
        scene=dict(
            xaxis=dict(title="UMAP 1", gridcolor=COLORS["grid"]),
            yaxis=dict(title="UMAP 2", gridcolor=COLORS["grid"]),
            zaxis=dict(title="UMAP 3", gridcolor=COLORS["grid"]),
            bgcolor=COLORS["bg"],
        ),
        showlegend=True,
        legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01),
    )
    
    return fig


# Interactive controls
color_options = ["ade_class", "is_anchor", "label_confidence"] + CLASSIFICATION_KEYS
color_dropdown = widgets.Dropdown(
    options=color_options,
    value="ade_class",
    description="Color by:",
    style={"description_width": "initial"},
)

output_3d = widgets.Output()

def update_3d_plot(change):
    with output_3d:
        output_3d.clear_output(wait=True)
        fig = create_3d_explorer(scenes, color_by=change["new"])
        display(fig)

color_dropdown.observe(update_3d_plot, names="value")

# Initial render
display(color_dropdown)
with output_3d:
    fig = create_3d_explorer(scenes, color_by="ade_class")
    display(fig)
display(output_3d)

Dropdown(description='Color by:', options=('ade_class', 'is_anchor', 'label_confidence', 'weather', 'time_of_d…

Output()

## 3. Boundary Analysis

### 3.1 Boundary Geography (Where are boundaries?)

In [4]:
def compute_boundary_density(scenes_df: pd.DataFrame, pairs_df: pd.DataFrame, 
                             key: str, resolution: int = 50) -> tuple:
    """Compute 2D density of boundary crossings for a semantic key."""
    # Get pairs for this key
    key_pairs = pairs_df[pairs_df["diff_key"] == key].copy()
    
    if len(key_pairs) == 0:
        return None, None, None
    
    # Get midpoints of each boundary crossing (in 2D UMAP space)
    idx_to_umap = {row["emb_index"]: (row["umap_x"], row["umap_y"]) 
                   for _, row in scenes_df.iterrows()}
    
    midpoints = []
    for _, row in key_pairs.iterrows():
        if row["idx_a"] in idx_to_umap and row["idx_b"] in idx_to_umap:
            xa, ya = idx_to_umap[row["idx_a"]]
            xb, yb = idx_to_umap[row["idx_b"]]
            midpoints.append(((xa + xb) / 2, (ya + yb) / 2))
    
    if len(midpoints) == 0:
        return None, None, None
    
    midpoints = np.array(midpoints)
    
    # Create 2D histogram
    x_range = (scenes_df["umap_x"].min() - 0.5, scenes_df["umap_x"].max() + 0.5)
    y_range = (scenes_df["umap_y"].min() - 0.5, scenes_df["umap_y"].max() + 0.5)
    
    density, x_edges, y_edges = np.histogram2d(
        midpoints[:, 0], midpoints[:, 1],
        bins=resolution,
        range=[x_range, y_range]
    )
    
    # Smooth the density
    density = ndimage.gaussian_filter(density, sigma=1.5)
    
    return density, x_edges, y_edges


def create_boundary_geography(scenes_df: pd.DataFrame, pairs_df: pd.DataFrame,
                              key: str = None) -> go.Figure:
    """Create 2D boundary geography map."""
    fig = go.Figure()
    
    # Background scatter of all points
    fig.add_trace(go.Scatter(
        x=scenes_df["umap_x"], y=scenes_df["umap_y"],
        mode="markers",
        marker=dict(size=3, color="#dddddd"),
        hoverinfo="skip",
        showlegend=False,
    ))
    
    # Compute and overlay boundary density
    keys_to_show = [key] if key else CLASSIFICATION_KEYS
    
    for i, k in enumerate(keys_to_show):
        density, x_edges, y_edges = compute_boundary_density(scenes_df, pairs_df, k)
        if density is None:
            continue
        
        x_centers = (x_edges[:-1] + x_edges[1:]) / 2
        y_centers = (y_edges[:-1] + y_edges[1:]) / 2
        
        fig.add_trace(go.Contour(
            z=density.T,
            x=x_centers,
            y=y_centers,
            colorscale="YlOrRd" if key else [[0, "rgba(255,255,255,0)"], [1, KEY_COLORS[i]]],
            opacity=0.6 if key else 0.3,
            showscale=key is not None,
            name=k,
            contours=dict(showlines=True, coloring="heatmap"),
        ))
    
    title = f"Boundary Geography: {key}" if key else "Boundary Crossing Hotspots (all keys)"
    fig.update_layout(
        **plotly_layout(title, height=500),
        xaxis=dict(title="UMAP 1", gridcolor=COLORS["grid"]),
        yaxis=dict(title="UMAP 2", gridcolor=COLORS["grid"], scaleanchor="x"),
    )
    
    return fig


# Interactive boundary geography
key_dropdown = widgets.Dropdown(
    options=["All keys"] + CLASSIFICATION_KEYS,
    value="All keys",
    description="Show:",
)

output_geo = widgets.Output()

def update_geo_plot(change):
    with output_geo:
        output_geo.clear_output(wait=True)
        key = None if change["new"] == "All keys" else change["new"]
        fig = create_boundary_geography(scenes, pairs, key=key)
        display(fig)

key_dropdown.observe(update_geo_plot, names="value")

display(key_dropdown)
with output_geo:
    fig = create_boundary_geography(scenes, pairs, key=None)
    display(fig)
display(output_geo)

Dropdown(description='Show:', options=('All keys', 'weather', 'time_of_day', 'depth_complexity', 'occlusion_le…

Output()

### 3.2 Boundary Sharpness (Fuzzy vs Crisp)

In [5]:
def compute_boundary_sharpness(scenes_df: pd.DataFrame, embeddings: np.ndarray,
                               key: str, k: int = 10) -> pd.DataFrame:
    """Compute per-scene boundary sharpness (% of k-NN with different label)."""
    from sklearn.neighbors import NearestNeighbors
    
    # Build k-NN index
    nn = NearestNeighbors(n_neighbors=k+1, metric="cosine")
    nn.fit(embeddings)
    
    distances, indices = nn.kneighbors(embeddings)
    
    # For each scene, compute % of neighbors with different label
    labels = scenes_df[key].values
    sharpness = []
    
    for i in range(len(scenes_df)):
        neighbor_labels = labels[indices[i, 1:]]  # Exclude self
        own_label = labels[i]
        diff_pct = np.mean(neighbor_labels != own_label)
        sharpness.append(diff_pct)
    
    return np.array(sharpness)


def create_sharpness_comparison() -> go.Figure:
    """Compare boundary sharpness across all keys."""
    fig = make_subplots(rows=2, cols=3, subplot_titles=CLASSIFICATION_KEYS)
    
    for i, key in enumerate(CLASSIFICATION_KEYS):
        row = i // 3 + 1
        col = i % 3 + 1
        
        sharpness = compute_boundary_sharpness(scenes, embeddings, key, k=10)
        
        fig.add_trace(
            go.Histogram(
                x=sharpness,
                nbinsx=20,
                marker_color=KEY_COLORS[i],
                opacity=0.7,
                name=key,
                showlegend=False,
            ),
            row=row, col=col
        )
        
        # Add annotation with mean
        fig.add_annotation(
            x=0.8, y=0.9,
            text=f"mean={sharpness.mean():.2f}",
            showarrow=False,
            xref=f"x{i+1 if i > 0 else ''} domain",
            yref=f"y{i+1 if i > 0 else ''} domain",
            font=dict(size=10),
        )
    
    fig.update_xaxes(title_text="Boundary sharpness (% diff neighbors)", range=[0, 1])
    fig.update_yaxes(title_text="Count")
    
    fig.update_layout(
        **plotly_layout("Boundary Sharpness Distribution (0=crisp, 1=fuzzy)", height=500),
    )
    
    return fig


# Note: This is computationally expensive, run once
print("Computing boundary sharpness (k=10 neighbors)...")
fig_sharpness = create_sharpness_comparison()
fig_sharpness.show()

Computing boundary sharpness (k=10 neighbors)...


### 3.3 Cost Surfaces (6-panel Small Multiples)

In [6]:
def create_cost_surface_grid() -> go.Figure:
    """Create 2x3 grid showing decision boundaries with ADE cost surface."""
    fig = make_subplots(
        rows=2, cols=3,
        subplot_titles=CLASSIFICATION_KEYS,
        horizontal_spacing=0.08,
        vertical_spacing=0.12,
    )
    
    # Filter scenes with ADE
    df_ade = scenes[scenes["ade"].notna()].copy()
    
    # Grid for interpolation
    x_min, x_max = scenes["umap_x"].min(), scenes["umap_x"].max()
    y_min, y_max = scenes["umap_y"].min(), scenes["umap_y"].max()
    grid_res = 40
    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, grid_res),
        np.linspace(y_min, y_max, grid_res)
    )
    grid_points = np.c_[xx.ravel(), yy.ravel()]
    
    # Interpolate ADE for background
    from scipy.interpolate import griddata
    ade_interp = griddata(
        df_ade[["umap_x", "umap_y"]].values,
        df_ade["ade"].values,
        grid_points,
        method="linear"
    ).reshape(xx.shape)
    
    for i, key in enumerate(CLASSIFICATION_KEYS):
        row = i // 3 + 1
        col = i % 3 + 1
        
        # ADE heatmap background
        fig.add_trace(
            go.Heatmap(
                z=ade_interp,
                x=np.linspace(x_min, x_max, grid_res),
                y=np.linspace(y_min, y_max, grid_res),
                colorscale="RdYlGn_r",
                showscale=(i == 2),  # Only show scale for one
                colorbar=dict(title="ADE", x=1.02) if i == 2 else None,
                opacity=0.6,
                hoverinfo="skip",
            ),
            row=row, col=col
        )
        
        # Fit k-NN classifier for decision boundary
        label_encoder = {v: j for j, v in enumerate(scenes[key].unique())}
        labels_encoded = scenes[key].map(label_encoder).values
        
        clf = KNeighborsClassifier(n_neighbors=5)
        clf.fit(scenes[["umap_x", "umap_y"]].values, labels_encoded)
        
        # Predict on grid
        proba = clf.predict_proba(grid_points)
        # Entropy as boundary indicator
        entropy = -np.sum(proba * np.log(proba + 1e-10), axis=1)
        entropy = entropy.reshape(xx.shape)
        
        # Boundary contour
        fig.add_trace(
            go.Contour(
                z=entropy,
                x=np.linspace(x_min, x_max, grid_res),
                y=np.linspace(y_min, y_max, grid_res),
                contours=dict(
                    start=entropy.max() * 0.5,
                    end=entropy.max(),
                    size=entropy.max() * 0.1,
                    coloring="lines",
                ),
                line=dict(color="#1a1a1a", width=1.5),
                showscale=False,
                hoverinfo="skip",
            ),
            row=row, col=col
        )
        
        # Data points
        fig.add_trace(
            go.Scatter(
                x=df_ade["umap_x"], y=df_ade["umap_y"],
                mode="markers",
                marker=dict(size=3, color="#1a1a1a", opacity=0.3),
                hoverinfo="skip",
                showlegend=False,
            ),
            row=row, col=col
        )
    
    fig.update_layout(
        **plotly_layout("Cost Surfaces with Decision Boundaries", height=600),
    )
    fig.update_xaxes(showticklabels=False)
    fig.update_yaxes(showticklabels=False)
    
    return fig


fig_cost = create_cost_surface_grid()
fig_cost.show()

## 4. Transition Analysis

### 4.1 Per-Key Transition Flows

In [7]:
def create_transition_sankey(pairs_df: pd.DataFrame, key: str) -> go.Figure:
    """Create Sankey diagram showing class transitions and their ADE impact."""
    # Filter to this key with ADE data
    df = pairs_df[(pairs_df["diff_key"] == key) & (pairs_df["rel_delta_ade"].notna())].copy()
    
    if len(df) == 0:
        return go.Figure().add_annotation(text=f"No data for {key}", showarrow=False)
    
    # Aggregate transitions
    trans = df.groupby(["value_a", "value_b"]).agg({
        "rel_delta_ade": "mean",
        "clip_a": "count",
        "ade_class_changed": "mean",
    }).reset_index()
    trans.columns = ["source", "target", "mean_ade", "count", "ade_change_rate"]
    
    # Get unique labels
    labels = list(set(trans["source"]) | set(trans["target"]))
    label_to_idx = {l: i for i, l in enumerate(labels)}
    
    # Node sizes (class frequency)
    node_counts = []
    for l in labels:
        count = len(df[(df["value_a"] == l) | (df["value_b"] == l)])
        node_counts.append(count)
    
    # Edge colors based on mean ΔADE (red=high, green=low)
    max_ade = trans["mean_ade"].max()
    edge_colors = []
    for ade in trans["mean_ade"]:
        # Map to color: green (low) -> yellow -> red (high)
        ratio = ade / max_ade if max_ade > 0 else 0
        if ratio < 0.5:
            r = int(255 * ratio * 2)
            g = 200
        else:
            r = 255
            g = int(200 * (1 - (ratio - 0.5) * 2))
        edge_colors.append(f"rgba({r},{g},100,0.6)")
    
    fig = go.Figure(go.Sankey(
        node=dict(
            pad=15,
            thickness=20,
            label=labels,
            color=[KEY_COLORS[i % len(KEY_COLORS)] for i in range(len(labels))],
        ),
        link=dict(
            source=[label_to_idx[s] for s in trans["source"]],
            target=[label_to_idx[t] for t in trans["target"]],
            value=trans["count"],
            color=edge_colors,
            customdata=trans[["mean_ade", "ade_change_rate"]].values,
            hovertemplate="%{source.label} -> %{target.label}<br>" +
                          "Count: %{value}<br>" +
                          "Mean ΔADE: %{customdata[0]:.3f}<br>" +
                          "ADE class change: %{customdata[1]:.1%}<extra></extra>",
        )
    ))
    
    fig.update_layout(
        **plotly_layout(f"Transition Flow: {key}", height=400),
    )
    
    return fig


# Interactive transition explorer
trans_dropdown = widgets.Dropdown(
    options=CLASSIFICATION_KEYS,
    value="depth_complexity",
    description="Key:",
)

output_trans = widgets.Output()

def update_trans_plot(change):
    with output_trans:
        output_trans.clear_output(wait=True)
        fig = create_transition_sankey(pairs, change["new"])
        display(fig)

trans_dropdown.observe(update_trans_plot, names="value")

display(trans_dropdown)
with output_trans:
    fig = create_transition_sankey(pairs, "depth_complexity")
    display(fig)
display(output_trans)

Dropdown(description='Key:', index=2, options=('weather', 'time_of_day', 'depth_complexity', 'occlusion_level'…

Output()

### 4.2 ADE Class Transition Matrix

In [8]:
def create_ade_transition_matrix(pairs_df: pd.DataFrame, key: str = None) -> go.Figure:
    """Create heatmap of ADE class transitions across boundary crossings."""
    df = pairs_df[pairs_df["rel_delta_ade"].notna()].copy()
    if key:
        df = df[df["diff_key"] == key]
    
    if len(df) == 0:
        return go.Figure().add_annotation(text="No data", showarrow=False)
    
    # Count transitions between ADE classes
    trans = df.groupby(["ade_class_a", "ade_class_b"]).size().unstack(fill_value=0)
    
    # Reorder to low, medium, high, critical
    order = [c for c in ADE_LABELS if c in trans.index]
    trans = trans.reindex(index=order, columns=order, fill_value=0)
    
    # Create annotations
    annotations = []
    for i, row_label in enumerate(trans.index):
        for j, col_label in enumerate(trans.columns):
            val = trans.iloc[i, j]
            annotations.append(dict(
                x=col_label, y=row_label,
                text=str(int(val)),
                showarrow=False,
                font=dict(color="white" if val > trans.values.max() / 2 else "black"),
            ))
    
    fig = go.Figure(go.Heatmap(
        z=trans.values,
        x=trans.columns.tolist(),
        y=trans.index.tolist(),
        colorscale="Blues",
        colorbar=dict(title="Count"),
    ))
    
    fig.update_layout(
        **plotly_layout(f"ADE Class Transitions{' (' + key + ')' if key else ''}", height=400),
        xaxis=dict(title="ADE Class B"),
        yaxis=dict(title="ADE Class A"),
        annotations=annotations,
    )
    
    return fig


# Show overall ADE transition matrix
fig_ade_trans = create_ade_transition_matrix(pairs)
fig_ade_trans.show()

# Compute ADE class change rates by key
print("\nADE Class Change Rates by Semantic Key:")
print("-" * 50)
for key in CLASSIFICATION_KEYS:
    df_key = pairs[(pairs["diff_key"] == key) & (pairs["rel_delta_ade"].notna())]
    if len(df_key) > 0:
        rate = df_key["ade_class_changed"].mean()
        print(f"{key:20s}: {rate:6.1%} ({len(df_key)} pairs)")


ADE Class Change Rates by Semantic Key:
--------------------------------------------------
weather             :  63.8% (437 pairs)
time_of_day         :  70.7% (140 pairs)
depth_complexity    :  64.4% (329 pairs)
occlusion_level     :  69.7% (429 pairs)
road_type           :  70.1% (572 pairs)
required_action     :  65.9% (745 pairs)


### 4.3 Danger Zone Identification

In [20]:
def identify_danger_zones(scenes_df: pd.DataFrame, embeddings: np.ndarray,
                          pairs_df: pd.DataFrame, k: int = 10,
                          boundary_threshold: float = 0.3) -> pd.DataFrame:
    """Identify scenes near boundaries that also have high ADE changes."""
    from sklearn.neighbors import NearestNeighbors
    
    # Compute boundary proximity for each key
    nn = NearestNeighbors(n_neighbors=k+1, metric="cosine")
    nn.fit(embeddings)
    _, indices = nn.kneighbors(embeddings)
    
    danger_scores = []
    
    for i, row in scenes_df.iterrows():
        clip_id = row["clip_id"]
        
        # Check if this scene is near a boundary for any key
        boundary_counts = {}
        for key in CLASSIFICATION_KEYS:
            labels = scenes_df[key].values
            own_label = labels[row["emb_index"]]
            neighbor_labels = labels[indices[row["emb_index"], 1:]]
            diff_pct = np.mean(neighbor_labels != own_label)
            boundary_counts[key] = diff_pct
        
        # Check ADE change rate for pairs involving this scene
        scene_pairs = pairs_df[
            ((pairs_df["clip_a"] == clip_id) | (pairs_df["clip_b"] == clip_id)) &
            (pairs_df["ade_class_changed"].notna())
        ]
        
        if len(scene_pairs) > 0:
            ade_change_rate = scene_pairs["ade_class_changed"].mean()
            mean_delta_ade = scene_pairs["rel_delta_ade"].mean()
        else:
            ade_change_rate = 0
            mean_delta_ade = 0
        
        # Max boundary proximity
        max_boundary = max(boundary_counts.values())
        boundary_key = max(boundary_counts, key=boundary_counts.get)
        
        # Danger score: near boundary AND causes ADE changes
        danger_score = max_boundary * ade_change_rate
        
        danger_scores.append({
            "clip_id": clip_id,
            "emb_index": row["emb_index"],
            "max_boundary_proximity": max_boundary,
            "boundary_key": boundary_key,
            "ade_change_rate": ade_change_rate,
            "mean_delta_ade": mean_delta_ade,
            "danger_score": danger_score,
            "is_danger": max_boundary > boundary_threshold and ade_change_rate > 0.5,
            "umap_x": row["umap_x"],
            "umap_y": row["umap_y"],
            "umap_z": row["umap_z"],
            "ade": row["ade"],
            "ade_class": row["ade_class"],
        })
    
    return pd.DataFrame(danger_scores)


def create_danger_zone_plot(danger_df: pd.DataFrame) -> go.Figure:
    """Visualize danger zones in 2D embedding space."""
    fig = go.Figure()
    
    # All points (faded)
    fig.add_trace(go.Scatter(
        x=danger_df["umap_x"], y=danger_df["umap_y"],
        mode="markers",
        marker=dict(
            size=5,
            color=danger_df["danger_score"],
            colorscale="YlOrRd",
            colorbar=dict(title="Danger Score"),
        ),
        text=danger_df.apply(
            lambda r: f"Boundary: {r['boundary_key']} ({r['max_boundary_proximity']:.1%})<br>" +
                      f"ADE change rate: {r['ade_change_rate']:.1%}<br>" +
                      f"Mean ΔADE: {r['mean_delta_ade']:.3f}",
            axis=1
        ),
        hoverinfo="text",
        name="All scenes",
    ))
    
    # Danger zones with glow effect
    danger_points = danger_df[danger_df["is_danger"]]
    if len(danger_points) > 0:
        # Glow (larger, transparent)
        fig.add_trace(go.Scatter(
            x=danger_points["umap_x"], y=danger_points["umap_y"],
            mode="markers",
            marker=dict(size=20, color="rgba(231, 76, 60, 0.3)"),
            hoverinfo="skip",
            showlegend=False,
        ))
        # Core
        fig.add_trace(go.Scatter(
            x=danger_points["umap_x"], y=danger_points["umap_y"],
            mode="markers",
            marker=dict(size=8, color="#e74c3c", line=dict(color="white", width=1)),
            name=f"Danger zones ({len(danger_points)})",
        ))
    
    fig.update_layout(
        **plotly_layout("Danger Zone Map", height=500),
        xaxis=dict(title="UMAP 1", gridcolor=COLORS["grid"]),
        yaxis=dict(title="UMAP 2", gridcolor=COLORS["grid"], scaleanchor="x"),
    )
    
    return fig


# Compute danger zones (this takes a moment)
print("Identifying danger zones...")
danger_df = identify_danger_zones(scenes, embeddings, pairs, k=10, boundary_threshold=0.98)

n_danger = danger_df["is_danger"].sum()
n_pairs_affected = len(pairs[
    (pairs["clip_a"].isin(danger_df[danger_df["is_danger"]]["clip_id"])) |
    (pairs["clip_b"].isin(danger_df[danger_df["is_danger"]]["clip_id"]))
])

print(f"\nSummary: {n_danger} scenes in danger zones")
print(f"         affecting {n_pairs_affected} pairs ({100*n_pairs_affected/len(pairs):.1f}% of total)")

fig_danger = create_danger_zone_plot(danger_df)
fig_danger.show()

Identifying danger zones...

Summary: 127 scenes in danger zones
         affecting 675 pairs (5.2% of total)


## 5. Pair Deep-Dive

### 5.1 Interactive Pair Explorer

In [10]:
def create_pair_scatter(pairs_df: pd.DataFrame, key: str = None,
                        min_delta: float = 0) -> go.Figure:
    """Create ADE_a vs ADE_b scatter plot."""
    df = pairs_df[pairs_df["rel_delta_ade"].notna()].copy()
    
    if key and key != "All":
        df = df[df["diff_key"] == key]
    
    if min_delta > 0:
        df = df[df["rel_delta_ade"] >= min_delta]
    
    if len(df) == 0:
        return go.Figure().add_annotation(text="No matching pairs", showarrow=False)
    
    # Color by diff_key
    fig = px.scatter(
        df,
        x="ade_a", y="ade_b",
        color="diff_key",
        color_discrete_sequence=KEY_COLORS,
        hover_data={
            "clip_a": True,
            "clip_b": True,
            "value_a": True,
            "value_b": True,
            "rel_delta_ade": ":.3f",
            "ade_class_a": True,
            "ade_class_b": True,
        },
        opacity=0.6,
    )
    
    # Add diagonal line
    max_val = max(df["ade_a"].max(), df["ade_b"].max()) * 1.1
    fig.add_trace(go.Scatter(
        x=[0, max_val], y=[0, max_val],
        mode="lines",
        line=dict(dash="dash", color=COLORS["grid"]),
        showlegend=False,
        hoverinfo="skip",
    ))
    
    # Add ADE class boundaries
    for threshold in [1.0, 2.5, 5.0]:
        fig.add_vline(x=threshold, line_dash="dot", line_color=COLORS["grid"], opacity=0.5)
        fig.add_hline(y=threshold, line_dash="dot", line_color=COLORS["grid"], opacity=0.5)
    
    fig.update_layout(
        **plotly_layout(f"Pair ADE Comparison ({len(df)} pairs)", height=500),
        xaxis=dict(title="ADE Scene A"),
        yaxis=dict(title="ADE Scene B"),
    )
    
    return fig


# Interactive pair explorer
pair_key_dropdown = widgets.Dropdown(
    options=["All"] + CLASSIFICATION_KEYS,
    value="All",
    description="Filter key:",
)

delta_slider = widgets.FloatSlider(
    value=0,
    min=0,
    max=2.0,
    step=0.1,
    description="Min ΔADE:",
)

output_pair = widgets.Output()

def update_pair_plot(change=None):
    with output_pair:
        output_pair.clear_output(wait=True)
        key = pair_key_dropdown.value if pair_key_dropdown.value != "All" else None
        fig = create_pair_scatter(pairs, key=key, min_delta=delta_slider.value)
        display(fig)

pair_key_dropdown.observe(update_pair_plot, names="value")
delta_slider.observe(update_pair_plot, names="value")

display(widgets.HBox([pair_key_dropdown, delta_slider]))
update_pair_plot()
display(output_pair)

Output()

### 5.2 Pair Connections in Embedding Space

In [11]:
def create_pair_connections(scenes_df: pd.DataFrame, pairs_df: pd.DataFrame,
                            key: str, max_pairs: int = 100) -> go.Figure:
    """Show pairs connected by lines in embedding space."""
    # Filter pairs
    df = pairs_df[(pairs_df["diff_key"] == key) & (pairs_df["rel_delta_ade"].notna())].copy()
    df = df.nlargest(max_pairs, "rel_delta_ade")  # Top by ΔADE
    
    if len(df) == 0:
        return go.Figure().add_annotation(text=f"No pairs for {key}", showarrow=False)
    
    # Build lookup
    clip_to_umap = {row["clip_id"]: (row["umap_x"], row["umap_y"])
                    for _, row in scenes_df.iterrows()}
    
    fig = go.Figure()
    
    # Background points
    fig.add_trace(go.Scatter(
        x=scenes_df["umap_x"], y=scenes_df["umap_y"],
        mode="markers",
        marker=dict(size=3, color="#dddddd"),
        hoverinfo="skip",
        showlegend=False,
    ))
    
    # Draw lines for pairs
    max_delta = df["rel_delta_ade"].max()
    for _, row in df.iterrows():
        if row["clip_a"] in clip_to_umap and row["clip_b"] in clip_to_umap:
            xa, ya = clip_to_umap[row["clip_a"]]
            xb, yb = clip_to_umap[row["clip_b"]]
            
            # Color by ΔADE
            intensity = row["rel_delta_ade"] / max_delta
            color = f"rgba(231, 76, 60, {0.3 + 0.5 * intensity})"
            
            fig.add_trace(go.Scatter(
                x=[xa, xb], y=[ya, yb],
                mode="lines",
                line=dict(color=color, width=1 + 2 * intensity),
                hoverinfo="text",
                text=f"{row['value_a']} -> {row['value_b']}<br>ΔADE: {row['rel_delta_ade']:.3f}",
                showlegend=False,
            ))
    
    # Highlight pair endpoints
    endpoints_x = []
    endpoints_y = []
    for _, row in df.iterrows():
        if row["clip_a"] in clip_to_umap and row["clip_b"] in clip_to_umap:
            xa, ya = clip_to_umap[row["clip_a"]]
            xb, yb = clip_to_umap[row["clip_b"]]
            endpoints_x.extend([xa, xb])
            endpoints_y.extend([ya, yb])
    
    fig.add_trace(go.Scatter(
        x=endpoints_x, y=endpoints_y,
        mode="markers",
        marker=dict(size=6, color="#e74c3c"),
        name=f"Pair endpoints ({len(df)} pairs)",
    ))
    
    fig.update_layout(
        **plotly_layout(f"Top {len(df)} High-ΔADE Pairs: {key}", height=500),
        xaxis=dict(title="UMAP 1", gridcolor=COLORS["grid"]),
        yaxis=dict(title="UMAP 2", gridcolor=COLORS["grid"], scaleanchor="x"),
    )
    
    return fig


# Show for most sensitive key
fig_connections = create_pair_connections(scenes, pairs, "depth_complexity", max_pairs=50)
fig_connections.show()

## 6. Summary & Key Findings

In [12]:
def create_summary_dashboard() -> go.Figure:
    """Create summary visualization of key findings."""
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=[
            "Sensitivity Ranking",
            "Trajectory Class Change Rate",
            "ADE Class Change Rate",
            "Boundary Sharpness",
        ],
        specs=[[{"type": "bar"}, {"type": "bar"}],
               [{"type": "bar"}, {"type": "bar"}]],
    )
    
    # 1. Sensitivity ranking
    ranking = summary["sensitivity_ranking"]
    keys = [r[0] for r in ranking]
    values = [r[1] for r in ranking]
    
    fig.add_trace(
        go.Bar(x=keys, y=values, marker_color=KEY_COLORS, showlegend=False),
        row=1, col=1
    )
    
    # 2. Trajectory class change rate
    traj_rates = [(k, stability[k].get("traj_change_rate", 0) or 0) for k in CLASSIFICATION_KEYS]
    traj_rates.sort(key=lambda x: x[1], reverse=True)
    
    fig.add_trace(
        go.Bar(
            x=[t[0] for t in traj_rates],
            y=[t[1] * 100 for t in traj_rates],
            marker_color=KEY_COLORS,
            showlegend=False,
        ),
        row=1, col=2
    )
    
    # 3. ADE class change rate
    ade_rates = []
    for key in CLASSIFICATION_KEYS:
        df_key = pairs[(pairs["diff_key"] == key) & (pairs["rel_delta_ade"].notna())]
        if len(df_key) > 0:
            rate = df_key["ade_class_changed"].mean()
            ade_rates.append((key, rate))
    ade_rates.sort(key=lambda x: x[1], reverse=True)
    
    fig.add_trace(
        go.Bar(
            x=[a[0] for a in ade_rates],
            y=[a[1] * 100 for a in ade_rates],
            marker_color=KEY_COLORS,
            showlegend=False,
        ),
        row=2, col=1
    )
    
    # 4. Boundary sharpness (mean % different neighbors)
    sharpness_means = []
    for key in CLASSIFICATION_KEYS:
        s = compute_boundary_sharpness(scenes, embeddings, key, k=10)
        sharpness_means.append((key, s.mean()))
    sharpness_means.sort(key=lambda x: x[1], reverse=True)
    
    fig.add_trace(
        go.Bar(
            x=[s[0] for s in sharpness_means],
            y=[s[1] * 100 for s in sharpness_means],
            marker_color=KEY_COLORS,
            showlegend=False,
        ),
        row=2, col=2
    )
    
    fig.update_yaxes(title_text="Rel ΔADE", row=1, col=1)
    fig.update_yaxes(title_text="% Changed", row=1, col=2)
    fig.update_yaxes(title_text="% Changed", row=2, col=1)
    fig.update_yaxes(title_text="% Diff Neighbors", row=2, col=2)
    
    fig.update_layout(
        **plotly_layout("Summary: Boundary Characteristics", height=600),
    )
    
    return fig


fig_summary = create_summary_dashboard()
fig_summary.show()